In [1]:
import inventorize as inv
import pandas as pd
from pulp import *


In [2]:
inbound= pd.read_excel('inbound.xlsx')

inbound= inbound.set_index('DCS')
demand= pd.read_excel('demand.xlsx')
demand= demand.set_index('shop')
outbound= pd.read_excel('outbound.xlsx')
outbound= outbound.set_index('DCS')
plant_costs= pd.read_excel('plant_costs.xlsx')
plant_costs= plant_costs.set_index('plant')
DC_costs= pd.read_excel('DC_costs.xlsx')
DC_costs= DC_costs.set_index('DCS')


In [3]:

plants= plant_costs.index
DCs= DC_costs.index
shops= demand.index

In [4]:
inbound_keys= [(w,p)for w in DCs for p in plants ]
outbound_keys= [(w,s)for w in DCs for s in shops]


In [5]:
inbound_var= LpVariable.dicts('inbound',inbound_keys,0,None,'Integer')
outbound_var= LpVariable.dicts('outbound',outbound_keys,0,None,'Integer')
open_dc= LpVariable.dicts('open_dc', DCs,cat='Binary')
open_plant=LpVariable.dicts('open_p',plants,cat='Binary')



In [6]:

inbound_cost= lpSum(inbound_var[(w,p)]* inbound.loc[w,p] for p in plants for w in DCs)
outbound_cost= lpSum(outbound_var[(w,s)]* outbound.loc[w,s] for w in DCs for s in shops)
production_cost= lpSum(inbound_var[(w,p)]* plant_costs.loc[p,'Var'] for p in plants for w in DCs)
opening_plant_cost= lpSum(open_plant[(p)]*plant_costs.loc[p,'fixed cost'] for p in plants)
opening_dc_cost= lpSum(open_dc[(w)]* DC_costs.loc[w,'Fixed Cost'] for w in DCs)
dc_operating_cost=lpSum(inbound_var[(w,p)]*DC_costs.loc[w,'Variable Cost'] for w in DCs for p in plants)


In [7]:
model= LpProblem('Extended',LpMinimize)


In [8]:
model += (inbound_cost +outbound_cost +production_cost+ opening_plant_cost+ opening_dc_cost+dc_operating_cost)


In [9]:

#### demand cosntratins

for s in shops :
    model+= lpSum(outbound_var[(w,s)]for w in DCs)>= demand.loc[s,'min_demand']
    model+= lpSum(outbound_var[(w,s)]for w in DCs)<= demand.loc[s,'max_demand']


    

#### capacity constraints

for p in plants:
    model+= lpSum(inbound_var[(w,p)]for w in DCs)<= lpSum(plant_costs.loc[p,'capacity']*open_plant[(p)])

for w in DCs:
    model+= lpSum(inbound_var[(w,p)] for p in plants)<= lpSum(DC_costs.loc[w,'capacity']*open_dc[(w)])


#### flow constraints 

for w in DCs:
     model += lpSum(inbound_var[(w,p)] for p in plants)== lpSum(outbound_var[(w,s)] for s in shops)


### opening constraints 

model += lpSum(open_dc[(w)] for w in DCs)<=10

model += lpSum(open_plant[(p)] for p in plants) >=1
model += lpSum(open_plant[(p)] for p in plants) <=3


In [10]:
model.solve()

Welcome to the CBC MILP Solver 
Version: 2.10.3 
Build Date: Dec 15 2019 

command line - /opt/anaconda3/lib/python3.9/site-packages/pulp/solverdir/cbc/osx/64/cbc /var/folders/25/6d_dng7s7d9c9vbyh1zlf3bm0000gn/T/d60516ea6e1740508c57c85a3fc74ce2-pulp.mps timeMode elapsed branch printingOptions all solution /var/folders/25/6d_dng7s7d9c9vbyh1zlf3bm0000gn/T/d60516ea6e1740508c57c85a3fc74ce2-pulp.sol (default strategy 1)
At line 2 NAME          MODEL
At line 3 ROWS
At line 67 COLUMNS
At line 1396 RHS
At line 1459 BOUNDS
At line 1683 ENDATA
Problem MODEL has 62 rows, 223 columns and 659 elements
Coin0008I MODEL read with 0 errors
Option for timeMode changed from cpu to elapsed
Continuous objective value is 7.25037e+06 - 0.00 seconds
Cgl0004I processed model has 42 rows, 223 columns (223 integer (13 of which binary)) and 466 elements
Cutoff increment increased from 1e-05 to 0.9999
Cbc0012I Integer solution of 7355109 found by DiveCoefficient after 0 iterations and 0 nodes (0.06 seconds)
Cbc001

1

In [11]:
[{v.name:v.varValue} for v in model.variables() if v.varValue  >0]    



[{"inbound_('DC_1',_'plant_1')": 9162.0},
 {"inbound_('DC_2',_'plant_2')": 24000.0},
 {"inbound_('DC_3',_'plant_2')": 12501.0},
 {"inbound_('DC_8',_'plant_1')": 10000.0},
 {'open_dc_DC_1': 1.0},
 {'open_dc_DC_2': 1.0},
 {'open_dc_DC_3': 1.0},
 {'open_dc_DC_8': 1.0},
 {'open_p_plant_1': 1.0},
 {'open_p_plant_2': 1.0},
 {"outbound_('DC_1',_'Shop_12')": 3038.0},
 {"outbound_('DC_1',_'Shop_14')": 2344.0},
 {"outbound_('DC_1',_'shop_2')": 3780.0},
 {"outbound_('DC_2',_'Shop_11')": 1713.0},
 {"outbound_('DC_2',_'Shop_16')": 1196.0},
 {"outbound_('DC_2',_'Shop_17')": 4174.0},
 {"outbound_('DC_2',_'Shop_4')": 3596.0},
 {"outbound_('DC_2',_'Shop_6')": 3473.0},
 {"outbound_('DC_2',_'Shop_7')": 2779.0},
 {"outbound_('DC_2',_'Shop_8')": 3319.0},
 {"outbound_('DC_2',_'Shop_9')": 3750.0},
 {"outbound_('DC_3',_'Shop_1')": 4129.0},
 {"outbound_('DC_3',_'Shop_11')": 1208.0},
 {"outbound_('DC_3',_'Shop_13')": 4219.0},
 {"outbound_('DC_3',_'Shop_3')": 249.0},
 {"outbound_('DC_3',_'Shop_5')": 2696.0},
 {"

In [12]:
len(model.variables())

223

In [13]:
sp=150


In [14]:
#### defining revenue
revenue=lpSum(inbound_var[(w,p)]* sp for w in DCs for p in plants)


In [15]:
##### profit_model

In [16]:
model2= LpProblem('Extended',LpMaximize)
model2 +=  revenue-(inbound_cost +outbound_cost +production_cost+ opening_plant_cost+ opening_dc_cost+dc_operating_cost)


In [17]:

#### demand cosntratins

for s in shops :
    model2+= lpSum(outbound_var[(w,s)]for w in DCs)>= demand.loc[s,'min_demand']
    model2+= lpSum(outbound_var[(w,s)]for w in DCs)<= demand.loc[s,'max_demand']


    

#### capacity constraints

for p in plants:
    model2+= lpSum(inbound_var[(w,p)]for w in DCs)<= lpSum(plant_costs.loc[p,'capacity']*open_plant[(p)])

for w in DCs:
    model2+= lpSum(inbound_var[(w,p)] for p in plants)<= lpSum(DC_costs.loc[w,'capacity']*open_dc[(w)])


#### flow constraints 

for w in DCs:
     model2 += lpSum(inbound_var[(w,p)] for p in plants)== lpSum(outbound_var[(w,s)] for s in shops)


### opening constraints 

model2 += lpSum(open_dc[(w)] for w in DCs)<=10
model2 += lpSum(open_dc[(w)] for w in DCs) >=1

model2 += lpSum(open_plant[(p)] for p in plants) >=1
model2 += lpSum(open_plant[(p)] for p in plants) <=3


In [18]:
model2.solve()

Welcome to the CBC MILP Solver 
Version: 2.10.3 
Build Date: Dec 15 2019 

command line - /opt/anaconda3/lib/python3.9/site-packages/pulp/solverdir/cbc/osx/64/cbc /var/folders/25/6d_dng7s7d9c9vbyh1zlf3bm0000gn/T/a5a273c8068241b68d1f839d6aac41e0-pulp.mps max timeMode elapsed branch printingOptions all solution /var/folders/25/6d_dng7s7d9c9vbyh1zlf3bm0000gn/T/a5a273c8068241b68d1f839d6aac41e0-pulp.sol (default strategy 1)
At line 2 NAME          MODEL
At line 3 ROWS
At line 68 COLUMNS
At line 1407 RHS
At line 1471 BOUNDS
At line 1695 ENDATA
Problem MODEL has 63 rows, 223 columns and 669 elements
Coin0008I MODEL read with 0 errors
Option for timeMode changed from cpu to elapsed
Continuous objective value is 1.41616e+06 - 0.00 seconds
Cgl0004I processed model has 43 rows, 223 columns (223 integer (13 of which binary)) and 476 elements
Cutoff increment increased from 1e-05 to 0.9999
Cbc0012I Integer solution of -1322898 found by DiveCoefficient after 0 iterations and 0 nodes (0.02 seconds)
C

1

In [19]:
[{v.name:v.varValue} for v in model2.variables() if v.varValue  >0]    


[{"inbound_('DC_1',_'plant_1')": 10000.0},
 {"inbound_('DC_2',_'plant_2')": 24000.0},
 {"inbound_('DC_3',_'plant_2')": 16000.0},
 {"inbound_('DC_5',_'plant_1')": 8989.0},
 {"inbound_('DC_8',_'plant_1')": 10000.0},
 {'open_dc_DC_1': 1.0},
 {'open_dc_DC_2': 1.0},
 {'open_dc_DC_3': 1.0},
 {'open_dc_DC_5': 1.0},
 {'open_dc_DC_8': 1.0},
 {'open_p_plant_1': 1.0},
 {'open_p_plant_2': 1.0},
 {"outbound_('DC_1',_'Shop_12')": 4050.0},
 {"outbound_('DC_1',_'Shop_14')": 910.0},
 {"outbound_('DC_1',_'shop_2')": 5040.0},
 {"outbound_('DC_2',_'Shop_11')": 3630.0},
 {"outbound_('DC_2',_'Shop_14')": 575.0},
 {"outbound_('DC_2',_'Shop_16')": 1196.0},
 {"outbound_('DC_2',_'Shop_17')": 4174.0},
 {"outbound_('DC_2',_'Shop_4')": 4795.0},
 {"outbound_('DC_2',_'Shop_6')": 4630.0},
 {"outbound_('DC_2',_'Shop_9')": 5000.0},
 {"outbound_('DC_3',_'Shop_1')": 5505.0},
 {"outbound_('DC_3',_'Shop_11')": 265.0},
 {"outbound_('DC_3',_'Shop_13')": 4219.0},
 {"outbound_('DC_3',_'Shop_3')": 2416.0},
 {"outbound_('DC_3',_

In [20]:
total_demand=sum(demand['max_demand'])
dist_matrix={ (w,s):1 if outbound.loc[w,s] <= 80 else 0 for w in DCs for s in shops}

current_service_level= lpSum(outbound_var[(w,s)].varValue* dist_matrix[(w,s)] for w in DCs for s in shops)/total_demand




In [21]:
current_service_level

0.7625277908778548

In [22]:
##### service level model

model3= LpProblem('Extended',LpMaximize)
model3 +=  revenue-(inbound_cost +outbound_cost +production_cost+ opening_plant_cost+ opening_dc_cost+dc_operating_cost)



#### demand cosntratins

for s in shops :
    model3+= lpSum(outbound_var[(w,s)]for w in DCs)>= demand.loc[s,'min_demand']
    model3+= lpSum(outbound_var[(w,s)]for w in DCs)<= demand.loc[s,'max_demand']


    

#### capacity constraints

for p in plants:
    model3+= lpSum(inbound_var[(w,p)]for w in DCs)<= lpSum(plant_costs.loc[p,'capacity']*open_plant[(p)])

for w in DCs:
    model3+= lpSum(inbound_var[(w,p)] for p in plants)<= lpSum(DC_costs.loc[w,'capacity']*open_dc[(w)])



#### flow constraints 

for w in DCs:
     model3 += lpSum(inbound_var[(w,p)] for p in plants)== lpSum(outbound_var[(w,s)] for s in shops)


### opening constraints 

model3 += lpSum(open_dc[(w)] for w in DCs)<=10
model3 += lpSum(open_dc[(w)] for w in DCs) >=1

model3 += lpSum(open_plant[(p)] for p in plants) >=1
model3 += lpSum(open_plant[(p)] for p in plants) <=3

    
    
    
    
    

##service level constraint

model3 +=  lpSum(outbound_var[(w,s)]* dist_matrix[(w,s)] for w in DCs for s in shops)/total_demand >= 0.85

model3.solve()








Welcome to the CBC MILP Solver 
Version: 2.10.3 
Build Date: Dec 15 2019 

command line - /opt/anaconda3/lib/python3.9/site-packages/pulp/solverdir/cbc/osx/64/cbc /var/folders/25/6d_dng7s7d9c9vbyh1zlf3bm0000gn/T/45a283850414413d8522922146a9e847-pulp.mps max timeMode elapsed branch printingOptions all solution /var/folders/25/6d_dng7s7d9c9vbyh1zlf3bm0000gn/T/45a283850414413d8522922146a9e847-pulp.sol (default strategy 1)
At line 2 NAME          MODEL
At line 3 ROWS
At line 69 COLUMNS
At line 1478 RHS
At line 1543 BOUNDS
At line 1767 ENDATA
Problem MODEL has 64 rows, 223 columns and 739 elements
Coin0008I MODEL read with 0 errors
Option for timeMode changed from cpu to elapsed
Continuous objective value is 1.35915e+06 - 0.00 seconds
Cgl0004I processed model has 44 rows, 223 columns (223 integer (13 of which binary)) and 546 elements
Cutoff increment increased from 1e-05 to 0.9999
Cbc0012I Integer solution of -1222567 found by DiveCoefficient after 0 iterations and 0 nodes (0.02 seconds)
C

1

In [23]:
[{v.name:v.varValue} for v in model3.variables() if v.varValue  >0]    



##model_relaxed.writeLP('exttended_model.txt')


[{"inbound_('DC_1',_'plant_1')": 10000.0},
 {"inbound_('DC_2',_'plant_2')": 24000.0},
 {"inbound_('DC_3',_'plant_2')": 16000.0},
 {"inbound_('DC_5',_'plant_1')": 5909.0},
 {"inbound_('DC_6',_'plant_2')": 5398.0},
 {"inbound_('DC_8',_'plant_1')": 10000.0},
 {'open_dc_DC_1': 1.0},
 {'open_dc_DC_2': 1.0},
 {'open_dc_DC_3': 1.0},
 {'open_dc_DC_5': 1.0},
 {'open_dc_DC_6': 1.0},
 {'open_dc_DC_8': 1.0},
 {'open_p_plant_1': 1.0},
 {'open_p_plant_2': 1.0},
 {"outbound_('DC_1',_'Shop_12')": 4050.0},
 {"outbound_('DC_1',_'Shop_14')": 910.0},
 {"outbound_('DC_1',_'shop_2')": 5040.0},
 {"outbound_('DC_2',_'Shop_11')": 3895.0},
 {"outbound_('DC_2',_'Shop_14')": 1434.0},
 {"outbound_('DC_2',_'Shop_16')": 1196.0},
 {"outbound_('DC_2',_'Shop_4')": 4795.0},
 {"outbound_('DC_2',_'Shop_5')": 829.0},
 {"outbound_('DC_2',_'Shop_6')": 4630.0},
 {"outbound_('DC_2',_'Shop_7')": 2221.0},
 {"outbound_('DC_2',_'Shop_9')": 5000.0},
 {"outbound_('DC_3',_'Shop_1')": 5505.0},
 {"outbound_('DC_3',_'Shop_13')": 4219.0}

In [24]:
current_service_level= lpSum(outbound_var[(w,s)].varValue* dist_matrix[(w,s)] for w in DCs for s in shops)/total_demand


In [25]:
binding= [{'constraint': name, 'slack': c.slack} for name,c in model3.constraints.items()]


In [26]:
plant_t= pd.read_excel('plant_transport.xlsx')

In [27]:
plant_t.set_index('plant',inplace=True)
####### direct dilevery model

In [28]:
plant_t

,Shop 1,shop 2,Shop 3,Shop 4,Shop 5,Shop 6,Shop 7,Shop 8,Shop 9,Shop 10,Shop 11,Shop 12,Shop 13,Shop 14,Shop 15,Shop 16,Shop 17,Shop 18
plant,,,,,,,,,,,,,,,,,,
plant 1,332,1200,1200,1200,1200,200,300,100,300,440,769,716,743,400,738,90,621,666
plant 2,1234,350,320,80,450,1434,70,420,304,220,300,950,977,314,972,324,111,300
plant 3,2234,2634,650,2634,2634,2634,300,2634,374,450,839,222,1720,394,1710,414,1476,1566


In [29]:
direct_keys= [(p,s)for p in plants for s in shops]

direct_var= LpVariable.dicts('direct',direct_keys,0,None,'Integer')


In [30]:
revenue=(lpSum(inbound_var[(w,p)]* sp for w in DCs for p in plants)+ 

lpSum(direct_var[(p,s)]* sp for p in plants for s in shops))


In [31]:
direct_cost=lpSum(direct_var[(p,s)]* plant_t.loc[p,s] for p in plants for s in shops)


In [32]:

model4= LpProblem('direct_del',LpMaximize)
model4 +=  revenue-(inbound_cost +outbound_cost +production_cost+ opening_plant_cost+ 
                    opening_dc_cost+dc_operating_cost+direct_cost)



#### demand cosntratins

for s in shops :
    model4+= (outbound_var[(w,s)]for w in DCs)+ lpSum(direct_var[(p,s)]for p in plants) >= demand.loc[s,'min_demand']
    model4+= (outbound_var[(w,s)]for w in DCs)+ lpSum(direct_var[(p,s)]for p in plants)<= demand.loc[s,'max_demand']


    

#### capacity constraints

for p in plants:
    model4+= lpSum(inbound_var[(w,p)]for w in DCs) + lpSum(direct_var[(p,s)]for s in shops) <= lpSum(plant_costs.loc[p,'capacity']*open_plant[(p)])

for w in DCs:
    model4+= lpSum(inbound_var[(w,p)] for p in plants)<= lpSum(DC_costs.loc[w,'capacity']*open_dc[(w)])



#### flow constraints 

for w in DCs:
     model4 += lpSum(inbound_var[(w,p)] for p in plants)== lpSum(outbound_var[(w,s)] for s in shops)


### opening constraints 

model4 += lpSum(open_dc[(w)] for w in DCs)<=10
model4 += lpSum(open_dc[(w)] for w in DCs) >=1

model4 += lpSum(open_plant[(p)] for p in plants) >=1
model4 += lpSum(open_plant[(p)] for p in plants) <=3

    
    
    
    
    

##service level constraint

#model3 +=  lpSum(outbound_var[(w,s)]* dist_matrix[(w,s)] for w in DCs for s in shops)/total_demand >= 0.85

model4.solve()





Welcome to the CBC MILP Solver 
Version: 2.10.3 
Build Date: Dec 15 2019 

command line - /opt/anaconda3/lib/python3.9/site-packages/pulp/solverdir/cbc/osx/64/cbc /var/folders/25/6d_dng7s7d9c9vbyh1zlf3bm0000gn/T/98c69785aeb845e79f58683fbe97d4af-pulp.mps max timeMode elapsed branch printingOptions all solution /var/folders/25/6d_dng7s7d9c9vbyh1zlf3bm0000gn/T/98c69785aeb845e79f58683fbe97d4af-pulp.sol (default strategy 1)
At line 2 NAME          MODEL
At line 3 ROWS
At line 68 COLUMNS
At line 1731 RHS
At line 1795 BOUNDS
At line 2073 ENDATA
Problem MODEL has 63 rows, 277 columns and 831 elements
Coin0008I MODEL read with 0 errors
Option for timeMode changed from cpu to elapsed
Continuous objective value is 2.28528e+06 - 0.00 seconds
Cgl0004I processed model has 43 rows, 277 columns (277 integer (13 of which binary)) and 584 elements
Cutoff increment increased from 1e-05 to 0.9999
Cbc0012I Integer solution of -2235640 found by DiveCoefficient after 0 iterations and 0 nodes (0.02 seconds)
C

1

In [33]:
[{v.name:v.varValue} for v in model4.variables() if v.varValue  >0]    


[{"direct_('plant_1',_'Shop_16')": 1595.0},
 {"direct_('plant_1',_'Shop_8')": 4425.0},
 {"direct_('plant_2',_'Shop_17')": 5565.0},
 {"direct_('plant_2',_'Shop_4')": 4795.0},
 {"direct_('plant_2',_'Shop_7')": 3705.0},
 {"inbound_('DC_1',_'plant_1')": 9090.0},
 {"inbound_('DC_2',_'plant_2')": 19464.0},
 {"inbound_('DC_3',_'plant_2')": 12140.0},
 {"inbound_('DC_8',_'plant_1')": 10000.0},
 {'open_dc_DC_1': 1.0},
 {'open_dc_DC_2': 1.0},
 {'open_dc_DC_3': 1.0},
 {'open_dc_DC_8': 1.0},
 {'open_p_plant_1': 1.0},
 {'open_p_plant_2': 1.0},
 {"outbound_('DC_1',_'Shop_12')": 4050.0},
 {"outbound_('DC_1',_'shop_2')": 5040.0},
 {"outbound_('DC_2',_'Shop_11')": 3895.0},
 {"outbound_('DC_2',_'Shop_14')": 2344.0},
 {"outbound_('DC_2',_'Shop_5')": 3595.0},
 {"outbound_('DC_2',_'Shop_6')": 4630.0},
 {"outbound_('DC_2',_'Shop_9')": 5000.0},
 {"outbound_('DC_3',_'Shop_1')": 5505.0},
 {"outbound_('DC_3',_'Shop_13')": 4219.0},
 {"outbound_('DC_3',_'Shop_3')": 2416.0},
 {"outbound_('DC_8',_'Shop_10')": 4270.0